# Clockwork T4 benchmark

Produces every performance number for the clockwork repo on a free Colab T4 GPU
(Runtime > Change runtime type > T4 GPU). Run top to bottom; each section states a rough
wall-clock range. Nothing is precomputed: every number printed comes from a command
executed in this run. Outputs land in results/, docs/figures/, and a results zip under
/content. Expect 3 to 6 hours total on the free tier.

## 1. GPU check

Confirms a CUDA device is visible via nvidia-smi and torch.cuda, and names it. Every
results table must carry this GPU name; the cell warns if the device is not a T4.
Expected wall clock: under 1 minute.

In [ ]:
import csv
import json
import os
import re
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch


def sh(cmd, echo=True):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        lines.append(line.rstrip("\n"))
        if echo:
            print(line, end="")
    proc.wait()
    return proc.returncode, lines


def must(cmd):
    rc, _ = sh(cmd)
    assert rc == 0, f"exit code {rc}: {' '.join(cmd)}"


try:
    rc, _ = sh(["nvidia-smi"])
except FileNotFoundError:
    rc = 1
assert rc == 0, "nvidia-smi failed: switch to a GPU runtime"
assert torch.cuda.is_available(), "torch sees no CUDA device: switch to a GPU runtime"
GPU_NAME = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
print(f"gpu: {GPU_NAME}")
print(f"memory: {gpu_props.total_memory / 1e9:.1f} GB")
print(f"capability: sm{gpu_props.major}{gpu_props.minor}")
if "T4" not in GPU_NAME:
    print(f"warning: this is not a T4; label every reported number with {GPU_NAME!r}")

## 2. Setup

Clones the repo, installs it editable with the gpu extra plus pytest, pytest-asyncio, and
pynvml (the bench runner samples GPU utilization through pynvml when present), points
HF_HOME under /content, and downloads Qwen/Qwen2.5-1.5B-Instruct. Expected wall clock:
5 to 15 minutes, mostly the model download.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO = BASE / "clockwork"
LOGS = BASE / "logs"
LOGS.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(BASE / "hf_home")
if not (REPO / "pyproject.toml").exists():
    must(["git", "clone", "https://github.com/jasonjesuraja06/clockwork", str(REPO)])
else:
    # A reused runtime keeps its old checkout; stale code must never run silently.
    must(["git", "-C", str(REPO), "pull", "--ff-only"])
sh(["git", "-C", str(REPO), "log", "--oneline", "-1"])
os.chdir(REPO)
must(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[gpu]"]
    + ["pytest", "pytest-asyncio", "pynvml"]
)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

In [ ]:
import transformers
from huggingface_hub import snapshot_download

print("model snapshot:", snapshot_download(MODEL))
print("torch", torch.__version__, "| transformers", transformers.__version__)

## 3. Correctness gates

All gates must pass before any benchmark cell runs, in this order: the slow Hugging Face
exact-match gate on the real model, a preflight proving Triton imports on CUDA (so the
gpu marker cannot skip silently), the gpu-marked Triton kernel tests, and the full
not-slow suite. A failed gate raises and stops the run. Note: the slow gate holds one
float32 copy of the 1.5B model in host RAM at a time (the Hugging Face reference is
freed before the engine loads), which fits the free tier. Expected wall clock:
20 to 60 minutes, dominated by the slow gate on Colab's CPU.

In [ ]:
gates = [
    (
        "hf exact match slow gate",
        [sys.executable, "-m", "pytest", "tests/test_hf_equivalence.py", "-q", "-m", "slow"],
    ),
    (
        "triton usable on this runtime",
        [
            sys.executable,
            "-c",
            "import torch; from clockwork.kernels.triton_paged_attn import HAS_TRITON; "
            "assert HAS_TRITON and torch.cuda.is_available()",
        ],
    ),
    (
        "triton kernel tests, gpu marker",
        [sys.executable, "-m", "pytest", "tests/test_triton_kernel.py", "-q", "-m", "gpu"],
    ),
    (
        "full suite, not slow",
        [sys.executable, "-m", "pytest", "-q", "-m", "not slow"],
    ),
]
GATE_RESULTS = []
for label, cmd in gates:
    t0 = time.monotonic()
    rc, _ = sh(cmd)
    took = time.monotonic() - t0
    assert rc == 0, f"gate failed: {label} (exit {rc})"
    GATE_RESULTS.append((label, " ".join(cmd), took))
    print(f"gate passed: {label} ({took:.0f}s)")
print("all correctness gates passed; benchmarks may run")

## 4. Paged decode kernel microbench

Times the Triton paged decode kernel against the torch fallback on identical tensors at
the served model's shapes (12 query heads, 2 kv heads, head_dim 128, block_size 16 from
the shipped config), batch 1 to 64, context 128 to 2048, float16. CUDA event timing with
warmup and torch.cuda.synchronize, plus a numerical cross-check per shape. The winner is
passed to the server in section 5, matching the resolve_backend policy of keeping the
faster backend. Expected wall clock: 1 to 3 minutes.

In [ ]:
from clockwork.kernels.attention import paged_attention_decode_torch, resolve_backend
from clockwork.kernels.triton_paged_attn import HAS_TRITON, triton_paged_attention_decode

assert HAS_TRITON, "triton did not import; rerun the setup cell"
NUM_HEADS, NUM_KV_HEADS, HEAD_DIM, BLOCK_SIZE = 12, 2, 128, 16
SCALE = HEAD_DIM**-0.5


def make_case(batch, ctx_len, dtype=torch.float16):
    blocks_per_seq = -(-ctx_len // BLOCK_SIZE)
    num_blocks = batch * blocks_per_seq
    shape = (num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_DIM)
    k_cache = torch.randn(shape, device="cuda", dtype=dtype)
    v_cache = torch.randn(shape, device="cuda", dtype=dtype)
    tables = torch.randperm(num_blocks, device="cuda").to(torch.int32)
    tables = tables.reshape(batch, blocks_per_seq)
    ctx_lens = torch.full((batch,), ctx_len, dtype=torch.int32, device="cuda")
    q = torch.randn(batch, NUM_HEADS, HEAD_DIM, device="cuda", dtype=dtype)
    return q, k_cache, v_cache, tables, ctx_lens


def time_ms(fn, args, warmup=10, iters=50):
    for _ in range(warmup):
        fn(*args)
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        fn(*args)
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / iters


torch.manual_seed(0)
MICRO_ROWS = []
for batch in (1, 4, 16, 64):
    for ctx_len in (128, 512, 2048):
        case = (*make_case(batch, ctx_len), SCALE)
        ref = paged_attention_decode_torch(*case)
        out = triton_paged_attention_decode(*case)
        torch.testing.assert_close(out, ref, atol=2e-3, rtol=2e-3)
        MICRO_ROWS.append(
            {
                "batch": batch,
                "ctx_len": ctx_len,
                "torch_ms": round(time_ms(paged_attention_decode_torch, case), 4),
                "triton_ms": round(time_ms(triton_paged_attention_decode, case), 4),
            }
        )
        del case, ref, out
torch.cuda.empty_cache()
for row in MICRO_ROWS:
    row["speedup"] = round(row["torch_ms"] / row["triton_ms"], 3)
print(f"{'batch':>6} {'ctx':>6} {'torch_ms':>10} {'triton_ms':>10} {'speedup':>8}")
for row in MICRO_ROWS:
    print(
        f"{row['batch']:>6} {row['ctx_len']:>6} {row['torch_ms']:>10.4f} "
        f"{row['triton_ms']:>10.4f} {row['speedup']:>8.3f}"
    )
total_torch = sum(row["torch_ms"] for row in MICRO_ROWS)
total_triton = sum(row["triton_ms"] for row in MICRO_ROWS)
BACKEND_WINNER = "triton" if total_triton <= total_torch else "torch"
print(
    f"decode winner on {GPU_NAME}: {BACKEND_WINNER} "
    f"(summed mean latency {total_triton:.3f} ms triton vs {total_torch:.3f} ms torch)"
)
print("resolve_backend('auto') here:", resolve_backend("auto"))
(REPO / "results").mkdir(exist_ok=True)
with (REPO / "results" / "microbench_decode.csv").open("w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(MICRO_ROWS[0]))
    writer.writeheader()
    writer.writerows(MICRO_ROWS)

## 4b. Kernel profiling and roofline

Profiles the paged decode kernel at the smallest and largest shapes of the microbench grid
above, on the backend section 4 selected, then classifies decode as memory bound or compute
bound from first principles. Three parts. The portable pass always runs: torch.profiler with
CPU and CUDA activities over 20 decode calls per shape, printing a key_averages table sorted
by self CUDA time. The nsight-compute pass runs only when ncu is on PATH; it profiles a
standalone decode script for achieved occupancy, dram throughput, and the compute and memory
pipe utilizations. Stock Colab runtimes ship no ncu, so that cell usually prints one line
saying so and continues. The roofline pass counts the KV bytes a decode step reads against
the FLOPs it performs, from the served model's own layer, head, and dtype numbers, measures
this device's copy bandwidth and fp16 matmul throughput for the machine balance, and prints
the verdict at every context length the microbench used. Every constant prints with its
source, and vendor peak figures print as stated datasheet constants rather than as
measurements. Tables land in results/: profile_decode.txt, profile_decode_ops.csv,
ncu_decode.txt, roofline_constants.csv, roofline_decode.csv, and
kernel_achieved_bandwidth.csv. Expected wall clock: 2 to 5 minutes, plus up to 10 more when
ncu is installed.

In [ ]:
from torch.profiler import ProfilerActivity, profile

# Reuses the microbench's shape grid and the backend section 4 selected, so the
# profile describes the kernel the server in section 5 will actually run.
PROF_DECODE_FN = (
    triton_paged_attention_decode if BACKEND_WINNER == "triton" else paged_attention_decode_torch
)
PROF_SHAPES = [("small", 1, 128), ("large", 64, 2048)]
PROF_ITERS = 20
RESULTS_DIR = REPO / "results"
RESULTS_DIR.mkdir(exist_ok=True)


def prof_self_cuda_us(event):
    # torch renamed self_cuda_time_total to self_device_time_total; accept either.
    for attr in ("self_device_time_total", "self_cuda_time_total"):
        value = getattr(event, attr, None)
        if value is not None:
            return float(value)
    return 0.0


def prof_table(prof, row_limit=12):
    for key in ("self_cuda_time_total", "self_device_time_total"):
        try:
            return prof.key_averages().table(sort_by=key, row_limit=row_limit)
        except (AssertionError, KeyError, ValueError, RuntimeError):
            continue
    return prof.key_averages().table(row_limit=row_limit)


torch.manual_seed(0)
PROF_TEXT = [f"paged decode profile on {GPU_NAME}, backend {BACKEND_WINNER}"]
PROF_OPS = []
for shape_label, prof_batch, prof_ctx in PROF_SHAPES:
    prof_case = (*make_case(prof_batch, prof_ctx), SCALE)
    for _ in range(10):  # warmup: triton jit compile, allocator, and cache warmup
        PROF_DECODE_FN(*prof_case)
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        for _ in range(PROF_ITERS):
            PROF_DECODE_FN(*prof_case)
        torch.cuda.synchronize()
    header = f"--- {shape_label}: batch {prof_batch}, ctx {prof_ctx}, {PROF_ITERS} calls ---"
    table = prof_table(prof)
    print(header)
    print(table)
    PROF_TEXT += [header, table]
    for event in prof.key_averages():
        self_us = prof_self_cuda_us(event)
        if self_us <= 0:
            continue
        calls = max(int(event.count), 1)
        PROF_OPS.append(
            {
                "shape": shape_label,
                "batch": prof_batch,
                "ctx_len": prof_ctx,
                "backend": BACKEND_WINNER,
                "op": event.key,
                "calls": calls,
                "self_cuda_us_total": round(self_us, 1),
                "self_cuda_us_per_call": round(self_us / calls, 3),
            }
        )
    del prof_case
torch.cuda.empty_cache()
PROF_SHAPE_ORDER = {label: index for index, (label, _, _) in enumerate(PROF_SHAPES)}
PROF_OPS.sort(key=lambda row: (PROF_SHAPE_ORDER[row["shape"]], -row["self_cuda_us_total"]))
(RESULTS_DIR / "profile_decode.txt").write_text("\n".join(PROF_TEXT) + "\n")
if PROF_OPS:
    with (RESULTS_DIR / "profile_decode_ops.csv").open("w", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(PROF_OPS[0]))
        writer.writeheader()
        writer.writerows(PROF_OPS)
    print("wrote results/profile_decode.txt and results/profile_decode_ops.csv")
else:
    print("wrote results/profile_decode.txt; no cuda events had self time, no ops csv")

In [ ]:
import shutil

# nsight-compute is absent from stock Colab runtimes, so this cell is best effort:
# it reports what it found and never raises into the rest of the run.
NCU_BATCH, NCU_CTX = 64, 2048
NCU_METRICS = [
    "sm__warps_active.avg.pct_of_peak_sustained_active",  # achieved occupancy
    "gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed",  # dram throughput
    "sm__throughput.avg.pct_of_peak_sustained_elapsed",  # compute pipe utilization
    "gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed",  # memory pipes
]
NCU_STATUS = "not-run"
NCU_OUTPUT = ""
try:
    ncu_path = shutil.which("ncu")
    ncu_script = LOGS / "ncu_decode_case.py"
    ncu_script.write_text(
        f'''"""Standalone paged decode launch, profiled by the nsight-compute pass."""

import torch

from clockwork.kernels.attention import paged_attention_decode_torch
from clockwork.kernels.triton_paged_attn import triton_paged_attention_decode

BATCH, CTX_LEN = {NCU_BATCH}, {NCU_CTX}
NUM_HEADS, NUM_KV_HEADS = {NUM_HEADS}, {NUM_KV_HEADS}
HEAD_DIM, BLOCK_SIZE = {HEAD_DIM}, {BLOCK_SIZE}
BACKEND = "{BACKEND_WINNER}"

torch.manual_seed(0)
blocks_per_seq = -(-CTX_LEN // BLOCK_SIZE)
num_blocks = BATCH * blocks_per_seq
shape = (num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_DIM)
k_cache = torch.randn(shape, device="cuda", dtype=torch.float16)
v_cache = torch.randn(shape, device="cuda", dtype=torch.float16)
tables = torch.randperm(num_blocks, device="cuda").to(torch.int32)
tables = tables.reshape(BATCH, blocks_per_seq)
ctx_lens = torch.full((BATCH,), CTX_LEN, dtype=torch.int32, device="cuda")
q = torch.randn(BATCH, NUM_HEADS, HEAD_DIM, device="cuda", dtype=torch.float16)
fn = triton_paged_attention_decode if BACKEND == "triton" else paged_attention_decode_torch
for _ in range(3):
    fn(q, k_cache, v_cache, tables, ctx_lens, HEAD_DIM**-0.5)
torch.cuda.synchronize()
print("decode launches done:", BACKEND, BATCH, CTX_LEN)
'''
    )
    ncu_version = ""
    if ncu_path is None:
        NCU_STATUS = "not-run: ncu is not on PATH (nsight-compute is absent on stock Colab)"
        NCU_OUTPUT = "no nsight-compute output: ncu was not installed on this runtime"
    else:
        ncu_cmd = [
            ncu_path,
            "--target-processes",
            "all",
            "--launch-count",
            "10",
            "--metrics",
            ",".join(NCU_METRICS),
            "--csv",
            "--page",
            "raw",
            sys.executable,
            str(ncu_script),
        ]
        ncu_version = subprocess.run(
            [ncu_path, "--version"], capture_output=True, text=True, timeout=120, check=False
        ).stdout.strip()
        print("ncu found at", ncu_path)
        print(ncu_version)
        print("running:", " ".join(ncu_cmd))
        done = subprocess.run(
            ncu_cmd, cwd=str(REPO), capture_output=True, text=True, timeout=1800, check=False
        )
        NCU_OUTPUT = (done.stdout + done.stderr).strip()
        NCU_STATUS = f"ran: exit {done.returncode}, {len(NCU_OUTPUT.splitlines())} output lines"
        if done.returncode != 0:
            NCU_STATUS += " (nonzero exit; counters often need elevated permissions)"
    print(f"NCU: {NCU_STATUS}")
    print(NCU_OUTPUT[:4000] if NCU_OUTPUT else "(no output)")
    (REPO / "results" / "ncu_decode.txt").write_text(
        f"gpu: {GPU_NAME}\nbackend: {BACKEND_WINNER}\n"
        f"shape: batch {NCU_BATCH}, ctx {NCU_CTX}\n"
        f"metrics: {','.join(NCU_METRICS)}\n"
        f"profiled script: {ncu_script}\n"
        f"ncu version: {ncu_version or 'n/a'}\n"
        f"status: {NCU_STATUS}\n\n{NCU_OUTPUT}\n"
    )
    print("wrote results/ncu_decode.txt")
except Exception as exc:
    NCU_STATUS = f"not-run: {exc}"
    print(f"NCU: {NCU_STATUS}")

In [ ]:
# Arithmetic intensity of decode, derived from the served model's own numbers.
# Every constant below prints with its source. The machine balance is measured
# in this cell; vendor peaks are printed as stated datasheet constants and are
# never presented as measurements.
hf_cfg = None
try:
    from transformers import AutoConfig

    hf_cfg = AutoConfig.from_pretrained(MODEL)
except Exception as exc:  # offline or an unexpected config layout
    NUM_LAYERS = 28
    LAYERS_SOURCE = f"documented layer count for {MODEL}; AutoConfig unavailable ({exc})"
if hf_cfg is not None:
    NUM_LAYERS = int(hf_cfg.num_hidden_layers)
    LAYERS_SOURCE = f"transformers AutoConfig({MODEL}).num_hidden_layers"
    assert int(hf_cfg.num_attention_heads) == NUM_HEADS, "microbench query heads vs model config"
    assert int(hf_cfg.num_key_value_heads) == NUM_KV_HEADS, "microbench kv heads vs model config"
    assert int(hf_cfg.hidden_size) // NUM_HEADS == HEAD_DIM, "microbench head_dim vs model config"
KV_BYTES_PER_ELEM = torch.empty((), dtype=torch.float16).element_size()


def measure_bandwidth_gb_s(elems=64 * 1024 * 1024, warmup=3, iters=20):
    """Device to device copy; each iteration reads the source and writes the destination."""
    src = torch.zeros(elems, dtype=torch.float16, device="cuda")
    dst = torch.empty_like(src)
    for _ in range(warmup):
        dst.copy_(src)
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        dst.copy_(src)
    end.record()
    torch.cuda.synchronize()
    ms = start.elapsed_time(end) / iters
    moved = 2 * src.numel() * src.element_size()
    del src, dst
    torch.cuda.empty_cache()
    return moved / (ms * 1e-3) / 1e9


def measure_fp16_tflops(n=4096, warmup=3, iters=20):
    """Square fp16 matmul; 2 * n^3 flops per call, the tensor core path on this device."""
    a = torch.randn(n, n, device="cuda", dtype=torch.float16)
    b = torch.randn(n, n, device="cuda", dtype=torch.float16)
    for _ in range(warmup):
        torch.matmul(a, b)
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        torch.matmul(a, b)
    end.record()
    torch.cuda.synchronize()
    ms = start.elapsed_time(end) / iters
    del a, b
    torch.cuda.empty_cache()
    return 2 * n**3 / (ms * 1e-3) / 1e12


BW_GB_S = measure_bandwidth_gb_s()
FP16_TFLOPS = measure_fp16_tflops()
RIDGE_FLOP_PER_BYTE = FP16_TFLOPS * 1e12 / (BW_GB_S * 1e9)


def kv_bytes_per_step(ctx_len, batch=1):
    """K and V for every layer of the context, read once per decode step."""
    return 2 * batch * NUM_LAYERS * ctx_len * NUM_KV_HEADS * HEAD_DIM * KV_BYTES_PER_ELEM


def attn_flops_per_step(ctx_len, batch=1):
    """Per layer and query head: 2*ctx*head_dim for q dot K, the same again for probs dot V."""
    return 2 * (2 * batch * NUM_LAYERS * NUM_HEADS * ctx_len * HEAD_DIM)


CONSTANTS = [
    ("num_layers", NUM_LAYERS, LAYERS_SOURCE),
    ("num_query_heads", NUM_HEADS, "section 4 microbench constant, from the served model"),
    ("num_kv_heads", NUM_KV_HEADS, "section 4 microbench constant, from the served model"),
    ("head_dim", HEAD_DIM, "section 4 microbench constant, from the served model"),
    ("kv_bytes_per_element", KV_BYTES_PER_ELEM, "torch float16 element_size()"),
    ("gpu_name", GPU_NAME, "torch.cuda.get_device_name(0)"),
    ("sm_count", gpu_props.multi_processor_count, "torch.cuda.get_device_properties(0)"),
    (
        "measured_bandwidth_gb_s",
        round(BW_GB_S, 1),
        "measured in this cell: 128 MiB fp16 device copy, read plus write",
    ),
    (
        "measured_fp16_tflops",
        round(FP16_TFLOPS, 2),
        "measured in this cell: 4096 cube fp16 matmul, 2*n^3 flops",
    ),
    (
        "measured_ridge_flop_per_byte",
        round(RIDGE_FLOP_PER_BYTE, 1),
        "measured_fp16_tflops divided by measured_bandwidth_gb_s",
    ),
]
print(f"{'constant':>28} {'value':>16}  source")
for name, value, source in CONSTANTS:
    print(f"{name:>28} {str(value):>16}  {source}")
print()

CTX_LENS = sorted({row["ctx_len"] for row in MICRO_ROWS})
ROOFLINE_ROWS = []
for ctx_len in CTX_LENS:
    moved = kv_bytes_per_step(ctx_len)
    flops = attn_flops_per_step(ctx_len)
    intensity = flops / moved
    bound = "memory-bound" if intensity < RIDGE_FLOP_PER_BYTE else "compute-bound"
    ROOFLINE_ROWS.append(
        {
            "ctx_len": ctx_len,
            "kv_bytes_per_step": moved,
            "attn_flops_per_step": flops,
            "intensity_flop_per_byte": round(intensity, 3),
            "ridge_flop_per_byte": round(RIDGE_FLOP_PER_BYTE, 1),
            "dram_time_ms": round(moved / (BW_GB_S * 1e9) * 1e3, 4),
            "math_time_ms": round(flops / (FP16_TFLOPS * 1e12) * 1e3, 4),
            "bound": bound,
        }
    )
# The ctx terms cancel: intensity is num_query_heads / num_kv_heads, the GQA group size.
assert all(
    abs(row["intensity_flop_per_byte"] - NUM_HEADS / NUM_KV_HEADS) < 1e-6 for row in ROOFLINE_ROWS
)
print("one decode step, one sequence, all layers of the served model:")
print(
    f"{'ctx':>6} {'kv MB read':>11} {'MFLOP':>9} {'flop/byte':>10} "
    f"{'dram ms':>9} {'math ms':>9} verdict"
)
for row in ROOFLINE_ROWS:
    print(
        f"{row['ctx_len']:>6} {row['kv_bytes_per_step'] / 1e6:>11.2f} "
        f"{row['attn_flops_per_step'] / 1e6:>9.1f} {row['intensity_flop_per_byte']:>10.2f} "
        f"{row['dram_time_ms']:>9.4f} {row['math_time_ms']:>9.4f} {row['bound']}"
    )
print(
    f"intensity {NUM_HEADS / NUM_KV_HEADS:.1f} flop/byte is num_query_heads / num_kv_heads, "
    "so it does not move with context length or batch"
)
print(
    f"measured ridge point {RIDGE_FLOP_PER_BYTE:.0f} flop/byte "
    f"({FP16_TFLOPS:.1f} TFLOP/s over {BW_GB_S:.0f} GB/s), "
    f"{RIDGE_FLOP_PER_BYTE / (NUM_HEADS / NUM_KV_HEADS):.0f}x above decode intensity"
)
print(
    "the matmul figure is the tensor core path; the decode kernel runs fp32 fma on the "
    "cuda cores, whose peak is lower, which only widens the gap"
)
print()

KERNEL_BW_ROWS = []
for row in MICRO_ROWS:
    batch, ctx_len = row["batch"], row["ctx_len"]
    # One attention layer, as timed in section 4: K and V for the context of every
    # sequence, plus the query read and the output write.
    kernel_bytes = (
        2 * batch * ctx_len * NUM_KV_HEADS * HEAD_DIM * KV_BYTES_PER_ELEM
        + 2 * batch * NUM_HEADS * HEAD_DIM * KV_BYTES_PER_ELEM
    )
    kernel_ms = row[f"{BACKEND_WINNER}_ms"]
    achieved = kernel_bytes / (kernel_ms * 1e-3) / 1e9
    KERNEL_BW_ROWS.append(
        {
            "batch": batch,
            "ctx_len": ctx_len,
            "backend": BACKEND_WINNER,
            "kernel_ms": kernel_ms,
            "bytes_moved": kernel_bytes,
            "achieved_gb_s": round(achieved, 1),
            "pct_of_measured_peak_bw": round(achieved / BW_GB_S * 100, 1),
        }
    )
print(f"{BACKEND_WINNER} decode kernel against the measured {BW_GB_S:.0f} GB/s ceiling:")
print(f"{'batch':>6} {'ctx':>6} {'ms':>9} {'GB/s':>9} {'pct of peak':>12}")
for row in KERNEL_BW_ROWS:
    print(
        f"{row['batch']:>6} {row['ctx_len']:>6} {row['kernel_ms']:>9.4f} "
        f"{row['achieved_gb_s']:>9.1f} {row['pct_of_measured_peak_bw']:>12.1f}"
    )
best_bw = max(KERNEL_BW_ROWS, key=lambda r: r["achieved_gb_s"])
grid_note = (
    f"the triton decode grid is batch x {NUM_HEADS} programs, so batch 1 occupies "
    f"{NUM_HEADS} of the {gpu_props.multi_processor_count} sms"
    if BACKEND_WINNER == "triton"
    else f"this device has {gpu_props.multi_processor_count} sms"
)
print(
    f"best: {best_bw['pct_of_measured_peak_bw']:.1f}% of that ceiling at batch "
    f"{best_bw['batch']}, ctx {best_bw['ctx_len']}; {grid_note}"
)
print(
    "no shape here reaches the dram roof, so the near-term limiter is occupancy and memory "
    "latency rather than bandwidth, which is what the ncu counters above report"
)
print()

if "T4" in GPU_NAME:
    # Stated vendor datasheet constants for the NVIDIA T4, not measured here.
    T4_PEAK_FP16_TFLOPS = 65.0
    T4_PEAK_BW_GB_S = 320.0
    t4_ridge = T4_PEAK_FP16_TFLOPS * 1e12 / (T4_PEAK_BW_GB_S * 1e9)
    print("stated NVIDIA T4 datasheet constants (vendor figures, not measurements):")
    print(f"  peak fp16 tensor throughput {T4_PEAK_FP16_TFLOPS:.0f} TFLOP/s")
    print(f"  peak memory bandwidth {T4_PEAK_BW_GB_S:.0f} GB/s")
    print(f"  datasheet ridge point {t4_ridge:.0f} flop/byte, same verdict as the measured one")
    CONSTANTS += [
        ("datasheet_peak_fp16_tflops", T4_PEAK_FP16_TFLOPS, "stated NVIDIA T4 datasheet constant"),
        ("datasheet_peak_bandwidth_gb_s", T4_PEAK_BW_GB_S, "stated NVIDIA T4 datasheet constant"),
        ("datasheet_ridge_flop_per_byte", round(t4_ridge, 1), "the two datasheet constants above"),
    ]
else:
    print(f"no datasheet constants printed: this run is on {GPU_NAME}, not a T4")


def write_rows(name, rows, fieldnames=None):
    path = REPO / "results" / name
    with path.open("w", newline="") as fh:
        if fieldnames is None:
            writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        else:
            writer = csv.writer(fh)
            writer.writerow(fieldnames)
            writer.writerows(rows)
    print("wrote", path.relative_to(REPO))


write_rows("roofline_constants.csv", CONSTANTS, fieldnames=["constant", "value", "source"])
write_rows("roofline_decode.csv", ROOFLINE_ROWS)
write_rows("kernel_achieved_bandwidth.csv", KERNEL_BW_ROWS)

## 4c. Real ShareGPT trace

The single-turn workloads in clockwork.bench read data/sharegpt.json when that file is
present and otherwise synthesize prompt and output lengths from a seeded lognormal
sampler over a small word list. That file is not in the checkout, so any run that skips
this cell measures SYNTHETIC traffic, not ShareGPT, and every row it produces has to say
so; that is why the shipped workloads carry a synthetic_ prefix.

This cell downloads the public ShareGPT V3 conversation dump from Hugging Face, prints
the resolved url, the byte count, and a sha256 prefix of exactly what arrived, and
records all of it in results/dataset_provenance.json. The benchmark reads a prefix of
that dump rather than the whole thing: the full file is about 650 MB and roughly 90000
conversations, which every run_bench invocation would reparse and retokenize inside the
free tier's RAM. Both the full-file hash and the hash of the file the benchmark reads
are recorded. The cell then replays one dataset-backed workload through the repo's own
generator, counts how many of its prompts appear verbatim in the trace file, and prints
the first one, which is what separates a dataset-derived run from the sampler. On
success those rows are dataset-derived and have to be renamed off the synthetic_ prefix,
which the cell prints; on any failure it says so loudly and the run continues on
synthetic data that must keep the synthetic label. Expected wall clock: 5 to 15 minutes.

In [ ]:
import hashlib

import clockwork.bench.workloads as bench_workloads
from clockwork.bench.configs import WORKLOADS

SHAREGPT_REPO = "anon8231489123/ShareGPT_Vicuna_unfiltered"
SHAREGPT_FILE = "ShareGPT_V3_unfiltered_cleaned_split.json"
SHAREGPT_KEEP = 12000
SHAREGPT_RULE = f"first {SHAREGPT_KEEP} conversations in file order with a human to gpt pair"
# workloads.py resolves the dataset relative to the installed package, so ask that module
# where it will look instead of assuming the checkout layout.
DATA_DIR = Path(getattr(bench_workloads, "_REPO_ROOT", REPO)) / "data"
SHAREGPT_PATH = DATA_DIR / "sharegpt.json"
SHAREGPT_SOURCE = DATA_DIR / "sharegpt_source.json"
SHAREGPT_URL = ""
SHAREGPT_SHA256 = ""
SHAREGPT_BYTES = 0
SHAREGPT_OK = False
SHAREGPT_REPLAYED = False
SHAREGPT_STATUS = "not-run"
SUBSET_SHA256 = ""
SUBSET_BYTES = 0
SOURCE_CONVERSATIONS = 0
SUBSET_CONVERSATIONS = 0
# Workload names change. The dataset-backed ones are whatever is left once the agent and
# ablation traces are removed, read from the config list at run time rather than typed here.
DATASET_WORKLOADS = [
    cfg.name for cfg in WORKLOADS if getattr(cfg, "kind", "") not in ("agent", "ablation")
]
if not DATASET_WORKLOADS:
    print("WARNING: clockwork.bench.configs.WORKLOADS lists no dataset-backed workload")

# Run against the file in a subprocess: the full dump is a multi-gigabyte object graph
# once parsed, and this kernel also holds torch.
SUBSET_SCRIPT = """
import json
import sys

src, dest, keep = sys.argv[1], sys.argv[2], int(sys.argv[3])
with open(src, encoding="utf-8") as fh:
    entries = json.load(fh)
kept = []
for entry in entries:
    turns = entry.get("conversations") or []
    if any(
        a.get("from") in ("human", "user")
        and b.get("from") in ("gpt", "assistant")
        and (a.get("value") or "")
        for a, b in zip(turns, turns[1:], strict=False)
    ):
        kept.append(entry)
    if len(kept) >= keep:
        break
with open(dest, "w", encoding="utf-8") as fh:
    json.dump(kept, fh)
print(len(entries), len(kept))
"""

# Replays one dataset-backed workload through the repo's own generator and checks the
# prompts against the trace file itself, which is what separates a dataset-derived run
# from the synthetic vocabulary sampler. Exits non-zero when nothing matched.
VERIFY_SCRIPT = """
import json
import sys

from transformers import AutoTokenizer

from clockwork.bench.configs import get_workload
from clockwork.bench.workloads import generate

reqs = generate(get_workload(sys.argv[1]), AutoTokenizer.from_pretrained(sys.argv[2]))
texts = []
for req in reqs:
    if getattr(req, "prompt", None):
        texts.append(req.prompt)
    else:
        texts.append((getattr(req, "messages", None) or [{}])[0].get("content", ""))
with open(sys.argv[3], encoding="utf-8") as fh:
    trace = json.load(fh)
pool = {
    turn.get("value")
    for entry in trace
    for turn in entry.get("conversations") or []
    if turn.get("value")
}
hits = sum(1 for text in texts if text in pool)
print("distinct prompts in the replayed trace:", len(set(texts)))
print(f"prompts found verbatim in the trace file: {hits} of {len(texts)}")
print("first prompt:", repr(texts[0][:160]))
sys.exit(0 if hits else 3)
"""


def hash_file(path):
    """Return (bytes, sha256) for a file, read in chunks so nothing large is held."""
    digest = hashlib.sha256()
    total = 0
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
            total += len(chunk)
    return total, digest.hexdigest()


def download(url, dest):
    """Stream url to dest, returning (bytes, sha256) of exactly what was written."""
    digest = hashlib.sha256()
    total = 0
    mark = 0
    req = urllib.request.Request(url, headers={"User-Agent": "clockwork-bench"})
    with urllib.request.urlopen(req, timeout=300) as resp, dest.open("wb") as fh:
        while True:
            chunk = resp.read(1 << 20)
            if not chunk:
                break
            fh.write(chunk)
            digest.update(chunk)
            total += len(chunk)
            if total - mark >= 128 << 20:
                mark = total
                print(f"  {total / 1e6:.0f} MB")
    return total, digest.hexdigest()


def resolve_url():
    """Resolve the dataset file name against the live repo listing, not a hardcoded guess."""
    names = []
    try:
        from huggingface_hub import HfApi

        names = list(HfApi().list_repo_files(SHAREGPT_REPO, repo_type="dataset"))
    except Exception as exc:
        print(f"could not list {SHAREGPT_REPO} ({exc}); assuming the standard file name")
    name = SHAREGPT_FILE
    if names and name not in names:
        found = sorted(n for n in names if n.endswith(".json") and "split" in n)
        if not found:
            raise RuntimeError(f"{SHAREGPT_FILE} is gone from {SHAREGPT_REPO} with no replacement")
        name = found[0]
        print(f"{SHAREGPT_FILE} is no longer in the repo; using {name}")
    return f"https://huggingface.co/datasets/{SHAREGPT_REPO}/resolve/main/{name}"


def usable(path):
    """Cheap sniff that path is the conversation list the workload generator expects."""
    if not path.is_file():
        return False
    with path.open("rb") as fh:
        head = fh.read(4096)
    return head.lstrip().startswith(b"[") and b"conversations" in head


dl_part = DATA_DIR / "sharegpt_source.part"
sub_part = DATA_DIR / "sharegpt_subset.part"
try:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    SHAREGPT_URL = resolve_url()
    print("url:", SHAREGPT_URL)
    if SHAREGPT_SOURCE.is_file():
        print("reusing the dump already in this checkout")
        SHAREGPT_BYTES, SHAREGPT_SHA256 = hash_file(SHAREGPT_SOURCE)
    else:
        SHAREGPT_BYTES, SHAREGPT_SHA256 = download(SHAREGPT_URL, dl_part)
        dl_part.replace(SHAREGPT_SOURCE)
    print(f"downloaded bytes: {SHAREGPT_BYTES}")
    print(f"downloaded sha256 prefix: {SHAREGPT_SHA256[:16]}")
    rc, lines = sh(
        [
            sys.executable,
            "-c",
            SUBSET_SCRIPT,
            str(SHAREGPT_SOURCE),
            str(sub_part),
            str(SHAREGPT_KEEP),
        ],
        echo=False,
    )
    if rc != 0:
        raise RuntimeError("the download is not the expected json: " + " ".join(lines[-3:]))
    SOURCE_CONVERSATIONS, SUBSET_CONVERSATIONS = (int(x) for x in lines[-1].split())
    sub_part.replace(SHAREGPT_PATH)
    SUBSET_BYTES, SUBSET_SHA256 = hash_file(SHAREGPT_PATH)
    SHAREGPT_OK = True
    SHAREGPT_STATUS = "ok"
    print(f"conversations in the dump: {SOURCE_CONVERSATIONS}")
    print(f"kept in {SHAREGPT_PATH}: {SUBSET_CONVERSATIONS} ({SHAREGPT_RULE})")
    print(f"kept bytes: {SUBSET_BYTES}, sha256 prefix: {SUBSET_SHA256[:16]}")
    print("dataset-backed workloads:", ", ".join(DATASET_WORKLOADS))
    if DATASET_WORKLOADS:
        verify = [sys.executable, "-c", VERIFY_SCRIPT, DATASET_WORKLOADS[0], MODEL]
        rc, lines = sh(verify + [str(SHAREGPT_PATH)], echo=False)
        for line in lines[-4:] if rc != 0 else lines[-3:]:
            print(line)
        SHAREGPT_REPLAYED = rc == 0
    if SHAREGPT_REPLAYED:
        stale = [name for name in DATASET_WORKLOADS if "synthetic" in name.lower()]
        if stale:
            print("NOTE: these workloads now sample the real trace, so their names and every")
            print("      published row for them must stop calling them synthetic:")
            print("      " + ", ".join(stale))
    else:
        print("!" * 84)
        print("The trace file is in place but no dataset-backed workload replayed it,")
        print("so the single-turn rows stay SYNTHETIC and must keep that label.")
        print("!" * 84)
except Exception as exc:
    dl_part.unlink(missing_ok=True)
    sub_part.unlink(missing_ok=True)
    SHAREGPT_OK = usable(SHAREGPT_PATH)
    SHAREGPT_STATUS = f"failed: {exc}"
    if SHAREGPT_OK:
        print(f"SHAREGPT: {exc}; the data/sharegpt.json already in this checkout stands")
    else:
        SHAREGPT_PATH.unlink(missing_ok=True)
        print("!" * 84)
        print(f"SHAREGPT DOWNLOAD FAILED: {exc}")
        print("The single-turn workloads fall back to SYNTHETIC lognormal lengths and")
        print("synthetic text. Every row from these workloads must be labeled SYNTHETIC")
        print("wherever it is published: " + (", ".join(DATASET_WORKLOADS) or "(none found)"))
        print("!" * 84)
if SHAREGPT_REPLAYED:
    SINGLE_TURN_DATA = "sharegpt"
elif SHAREGPT_OK:
    SINGLE_TURN_DATA = "sharegpt file present but the replay check did not confirm it"
else:
    SINGLE_TURN_DATA = "SYNTHETIC"
(REPO / "results").mkdir(parents=True, exist_ok=True)
(REPO / "results" / "dataset_provenance.json").write_text(
    json.dumps(
        {
            "status": SHAREGPT_STATUS,
            "single_turn_data": SINGLE_TURN_DATA,
            "repo": SHAREGPT_REPO,
            "url": SHAREGPT_URL,
            "downloaded_bytes": SHAREGPT_BYTES,
            "downloaded_sha256": SHAREGPT_SHA256,
            "conversations_downloaded": SOURCE_CONVERSATIONS,
            "benchmark_file": str(SHAREGPT_PATH),
            "benchmark_file_bytes": SUBSET_BYTES,
            "benchmark_file_sha256": SUBSET_SHA256,
            "benchmark_file_rule": SHAREGPT_RULE,
            "conversations_used": SUBSET_CONVERSATIONS,
            "dataset_backed_workloads": DATASET_WORKLOADS,
        },
        indent=2,
    )
    + "\n"
)
print("provenance:", REPO / "results" / "dataset_provenance.json")
print("single-turn data for this run:", SINGLE_TURN_DATA)

## 5. Clockwork benchmark

Launches scripts/serve.py as a subprocess in float16 (the T4 is sm75, which has no usable
bfloat16), waits for /health, prints /stats to show the attention backend actually in
use, then drives scripts/run_bench.py. The KV cache num_blocks arithmetic for the 16 GB
card is justified in the code comment. The radix-off ablation workloads are replayed
against a second server started with the prefix cache disabled, so every workload row in
results/clockwork/summary.csv is measured under the setting its name claims. Expected
wall clock: 40 to 120 minutes.

In [ ]:
def wait_health(url, proc, timeout_s):
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"server exited early with code {proc.returncode}")
        try:
            with urllib.request.urlopen(url, timeout=5) as resp:
                if resp.status == 200:
                    return
        except Exception:
            time.sleep(2.0)
    raise TimeoutError(f"{url} not healthy within {timeout_s}s")


def start_server(cmd, log_path, health_url, timeout_s):
    fd = os.open(str(log_path), os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o644)
    proc = subprocess.Popen(cmd, stdout=fd, stderr=subprocess.STDOUT)
    os.close(fd)
    print("launched:", " ".join(cmd))
    try:
        wait_health(health_url, proc, timeout_s)
    except Exception:
        print(Path(log_path).read_text(errors="replace")[-4000:])
        if proc.poll() is None:
            proc.kill()
        raise
    return proc


def stop_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait(timeout=30)


# One KV block holds k and v (2 tensors) x 28 layers x block_size 16 x 2 kv heads
# x head_dim 128 x 2 bytes fp16 = 458752 bytes. The scheduler ceiling is
# max_num_seqs 64 x max_model_len 4096 = 262144 tokens = 16384 blocks (7.5 GB);
# 2048 extra blocks keep radix prefixes resident after release, so num_blocks
# 18432 = 8.5 GB. With the fp16 weights (about 3.1 GB) plus CUDA context and
# activations, that fits a 16 GB T4 with headroom.
BYTES_PER_BLOCK = 2 * 28 * BLOCK_SIZE * NUM_KV_HEADS * HEAD_DIM * 2
assert BYTES_PER_BLOCK == 458752
NUM_BLOCKS = 18432
kv_gb = NUM_BLOCKS * BYTES_PER_BLOCK / 1e9
print(f"kv cache: {NUM_BLOCKS} blocks x {BYTES_PER_BLOCK} B per block = {kv_gb:.2f} GB")
CLOCKWORK_URL = "http://127.0.0.1:8000"
clockwork_cmd = [
    sys.executable,
    "scripts/serve.py",
    "--config",
    "configs/qwen2.5-1.5b-instruct.yaml",
    "--port",
    "8000",
    "--set",
    "device=cuda",
    "--set",
    "dtype=float16",
    "--set",
    f"num_blocks={NUM_BLOCKS}",
    "--set",
    f"attention_backend={BACKEND_WINNER}",
]
clockwork_proc = start_server(
    clockwork_cmd, LOGS / "clockwork_server.log", f"{CLOCKWORK_URL}/health", timeout_s=900
)
with urllib.request.urlopen(f"{CLOCKWORK_URL}/stats", timeout=10) as resp:
    print(resp.read().decode("utf-8"))

In [ ]:
from clockwork.bench.configs import WORKLOADS

radix_on_names = ",".join(cfg.name for cfg in WORKLOADS if cfg.radix_enabled)
radix_off_names = ",".join(cfg.name for cfg in WORKLOADS if not cfg.radix_enabled)
bench_on_cmd = [
    sys.executable,
    "scripts/run_bench.py",
    "--configs",
    radix_on_names,
    "--base-url",
    CLOCKWORK_URL,
    "--out",
    "results/clockwork",
]
must(bench_on_cmd)
stop_server(clockwork_proc)
# The radix-off traces replay against a server whose prefix cache is disabled,
# appending to the same summary.csv so every workload has exactly one measured row.
clockwork_off_proc = start_server(
    clockwork_cmd + ["--set", "enable_prefix_cache=false"],
    LOGS / "clockwork_server_radix_off.log",
    f"{CLOCKWORK_URL}/health",
    timeout_s=900,
)
bench_off_cmd = [
    sys.executable,
    "scripts/run_bench.py",
    "--configs",
    radix_off_names,
    "--base-url",
    CLOCKWORK_URL,
    "--out",
    "results/clockwork",
]
must(bench_off_cmd)
stop_server(clockwork_off_proc)
CLOCKWORK_BENCH_CMDS = [" ".join(bench_on_cmd), " ".join(bench_off_cmd)]

## 5b. Derived-metric experiments

Two extra measurements on a fresh clockwork server. First, the same agent workload runs
twice, a cold pass on an empty prefix cache and an identical warm replay, so the TTFT
delta isolates the prefix cache effect on identical requests under an identical
schedule. Second, three agent workloads rerun with fixed-length generation
(ignore_eos, max_tokens 64) on clockwork here and on vLLM in its section below;
identical output token counts make session completion latency comparable across
engines. Expected wall clock: 10 to 20 minutes.


In [ ]:
EXPERIMENT_TTFT_WL = "agent_p1536_t6to12_pois_r2"
FIXEDLEN_WLS = "agent_p1024_t4to8_pois_r2,agent_p1024_t4to8_pois_r4,agent_p1536_t6to12_pois_r2"
exp_proc = start_server(
    clockwork_cmd,
    LOGS / "clockwork_server_experiments.log",
    f"{CLOCKWORK_URL}/health",
    timeout_s=900,
)
for out in ("results/ttft_cold", "results/ttft_warm"):
    must(
        [
            sys.executable,
            "scripts/run_bench.py",
            "--configs",
            EXPERIMENT_TTFT_WL,
            "--base-url",
            CLOCKWORK_URL,
            "--out",
            out,
        ]
    )
must(
    [
        sys.executable,
        "scripts/run_bench.py",
        "--configs",
        FIXEDLEN_WLS,
        "--base-url",
        CLOCKWORK_URL,
        "--out",
        "results/sessions_fixed/clockwork",
        "--max-tokens",
        "64",
        "--ignore-eos",
    ]
)
stop_server(exp_proc)

## 6. vLLM baseline

Installs vLLM into this environment only if pip's resolver would not replace the
installed torch or transformers, otherwise into a fresh venv; the cell prints which happened. When venv cannot
bootstrap pip (Colab system python omits ensurepip), the cell installs python3-venv
via apt or falls back to virtualenv; if vLLM still cannot install, the baseline is
reported as not-run and the rest of the notebook continues. Serves the same model in float16 on its default settings, printed verbatim from
the server's own startup log, then runs the same scripts/run_bench.py workloads against
its port. Expected wall clock: 30 to 90 minutes including the install.

The benchmark loop runs one run_bench invocation per workload and snapshots the
Prometheus /metrics prefix cache counters of vLLM around each one, so the per
workload prefix hit rate of vLLM is measured from counter deltas
(results/vllm/hitrates.csv, folded into its summary.csv). The clockwork config
admits prompts up to max_num_batched_tokens 4096, so all 21 workloads produce data.


In [ ]:
def make_env(path):
    if sh([sys.executable, "-m", "venv", str(path)])[0] == 0:
        return True
    # colab's debian python omits ensurepip, so venv cannot bootstrap pip
    ver = f"python{sys.version_info.major}.{sys.version_info.minor}-venv"
    sh(["apt-get", "update", "-qq"])
    sh(["apt-get", "install", "-qq", "-y", ver, "python3-venv"])
    if sh([sys.executable, "-m", "venv", str(path)])[0] == 0:
        return True
    if sh([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])[0] == 0:
        return sh([sys.executable, "-m", "virtualenv", "-q", str(path)])[0] == 0
    return False


vllm_python = None
rc, dry_lines = sh([sys.executable, "-m", "pip", "install", "--dry-run", "vllm"], echo=False)
would = next((line for line in dry_lines if line.startswith("Would install")), "")
conflict = re.search(r"\b(torch|transformers)-\d", would)
ok = rc == 0 and not conflict
if ok and sh([sys.executable, "-m", "pip", "install", "-q", "vllm"])[0] == 0:
    vllm_python = sys.executable
    print("VLLM: pins compatible, installed in the current environment")
if vllm_python is None:
    if rc != 0:
        reason = "pip resolution failed"
    elif conflict:
        reason = f"pip would replace {conflict.group(1)}"
    else:
        reason = "direct install failed"
    env_dir = BASE / "venv_vllm"
    if make_env(env_dir):
        candidate = str(env_dir / "bin" / "python")
        if sh([candidate, "-m", "pip", "install", "-q", "vllm"])[0] == 0:
            vllm_python = candidate
            print(f"VLLM: {reason}, installed in a fresh venv")
if vllm_python is None:
    print("VLLM: not-run (install failed; see output above)")

In [ ]:
VLLM_BENCH_CMD = None
if vllm_python is None:
    print("VLLM: not-run, baseline skipped")
else:
    VLLM_URL = "http://127.0.0.1:8100"
    vllm_bin = Path(vllm_python).parent / "vllm"
    vllm_cmd = [str(vllm_bin), "serve", MODEL]
    if not vllm_bin.exists():
        vllm_cmd = [vllm_python, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL]
    vllm_cmd += ["--dtype", "float16", "--port", "8100"]
    health = f"{VLLM_URL}/health"
    vllm_proc = start_server(vllm_cmd, LOGS / "vllm_server.log", health, timeout_s=1800)
    print("vllm settings, verbatim from the server log:")
    log_lines = (LOGS / "vllm_server.log").read_text(errors="replace").splitlines()
    shown = [line for line in log_lines if "Namespace(" in line or "EngineArgs(" in line]
    for line in shown or log_lines[:30]:
        print(line)
    import csv
    import urllib.request

    from clockwork.bench.configs import WORKLOADS

    def scrape_prefix_counters(base):
        # vllm v1 exports vllm:prefix_cache_queries and vllm:prefix_cache_hits
        # token counters at /metrics; sum across label sets, tolerate absence.
        try:
            text = urllib.request.urlopen(base + "/metrics", timeout=10).read().decode()
        except Exception:
            return None
        hits, queries, found = 0.0, 0.0, False
        for line in text.splitlines():
            if not line.startswith("vllm:") or "prefix_cache" not in line:
                continue
            name = line.split("{")[0].split(" ")[0]
            try:
                value = float(line.rsplit(" ", 1)[1])
            except ValueError:
                continue
            if "hit" in name:
                hits, found = hits + value, True
            elif "quer" in name:
                queries, found = queries + value, True
        return (hits, queries) if found else None

    hit_rows = []
    for wl in WORKLOADS:
        before = scrape_prefix_counters(VLLM_URL)
        must(
            [
                sys.executable,
                "scripts/run_bench.py",
                "--configs",
                wl.name,
                "--base-url",
                VLLM_URL,
                "--out",
                "results/vllm",
            ]
        )
        after = scrape_prefix_counters(VLLM_URL)
        if before is None or after is None:
            hit_rows.append((wl.name, "", "", ""))
            continue
        dh, dq = after[0] - before[0], after[1] - before[1]
        hit_rows.append((wl.name, dh, dq, dh / dq if dq > 0 else ""))
    must(
        [
            sys.executable,
            "scripts/run_bench.py",
            "--configs",
            FIXEDLEN_WLS,
            "--base-url",
            VLLM_URL,
            "--out",
            "results/sessions_fixed/vllm",
            "--max-tokens",
            "64",
            "--ignore-eos",
        ]
    )
    stop_server(vllm_proc)
    with (REPO / "results" / "vllm" / "hitrates.csv").open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["workload", "prefix_hit_tokens", "prefix_query_tokens", "hit_rate"])
        writer.writerows(hit_rows)
    # Each run_bench invocation rewrites summary.csv with only its own workload,
    # so rebuild the full summary from the per-request CSVs, then fold in the
    # scraped hit rates so every downstream table and figure carries them.
    (REPO / "results" / "vllm" / "summary.csv").unlink(missing_ok=True)
    must([sys.executable, "scripts/collect_results.py", "--out", "results/vllm", "--no-figures"])
    rates = {name: rate for name, _, _, rate in hit_rows if rate != ""}
    summary_path = REPO / "results" / "vllm" / "summary.csv"
    rows = list(csv.DictReader(summary_path.open()))
    for row in rows:
        if row["workload"] in rates:
            row["hit_rate"] = f"{rates[row['workload']]:.4f}"
    with summary_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    scraped = sum(1 for r in hit_rows if r[3] != "")
    print(f"vllm hit rates scraped for {scraped} of {len(hit_rows)} workloads")
    VLLM_BENCH_CMD = (
        "scripts/run_bench.py --configs <name> --base-url " + VLLM_URL + " --out results/vllm"
        " per workload, hit rate from /metrics prefix counter deltas"
    )

In [ ]:
import csv


def _ttfts(path):
    return sorted(
        float(r["ttft_ms"]) for r in csv.DictReader(path.open()) if r["ttft_ms"] and not r["error"]
    )


def _pct(vals, p):
    i = max(0, int(round(p / 100 * len(vals) + 0.5)) - 1)
    return vals[min(i, len(vals) - 1)]


derived = []
cold = _ttfts(REPO / "results" / "ttft_cold" / f"{EXPERIMENT_TTFT_WL}.csv")
warm = _ttfts(REPO / "results" / "ttft_warm" / f"{EXPERIMENT_TTFT_WL}.csv")
if cold and warm:
    for p in (50, 99):
        cut = (1 - _pct(warm, p) / _pct(cold, p)) * 100
        derived += [
            (f"ttft_p{p}_cold_ms", f"{_pct(cold, p):.1f}"),
            (f"ttft_p{p}_warm_ms", f"{_pct(warm, p):.1f}"),
            (f"ttft_p{p}_cut_pct", f"{cut:.1f}"),
        ]
        print(
            f"prefix cache ttft p{p}: cold {_pct(cold, p):.0f} ms, "
            f"warm {_pct(warm, p):.0f} ms, cut {cut:.0f}%"
        )


def _sessions(path):
    lo, hi, bad = {}, {}, 0
    for r in csv.DictReader(path.open()):
        if r["error"]:
            bad += 1
            continue
        k = r["session_id"]
        lo[k] = min(lo.get(k, 1e18), float(r["arrival_s"]))
        hi[k] = max(hi.get(k, 0.0), float(r["end_s"]))
    return {k: hi[k] - lo[k] for k in lo}, bad


totals = [0.0, 0.0, 0]
for name in FIXEDLEN_WLS.split(","):
    cpath = REPO / "results" / "sessions_fixed" / "clockwork" / f"{name}.csv"
    vpath = REPO / "results" / "sessions_fixed" / "vllm" / f"{name}.csv"
    if not (cpath.exists() and vpath.exists()):
        print(f"{name}: fixed-length pair incomplete, skipped")
        continue
    ct, cbad = _sessions(cpath)
    vt, vbad = _sessions(vpath)
    if cbad or vbad:
        print(f"{name}: {cbad} clockwork and {vbad} vllm errored requests excluded")
    common = sorted(set(ct) & set(vt))
    if not common:
        continue
    cm = sum(ct[k] for k in common) / len(common)
    vm = sum(vt[k] for k in common) / len(common)
    totals[0] += sum(ct[k] for k in common)
    totals[1] += sum(vt[k] for k in common)
    totals[2] += len(common)
    print(
        f"{name}: mean session clockwork {cm:.2f}s vllm {vm:.2f}s, "
        f"change {(1 - cm / vm) * 100:+.0f}%"
    )
    derived += [
        (f"session_mean_s_clockwork_{name}", f"{cm:.3f}"),
        (f"session_mean_s_vllm_{name}", f"{vm:.3f}"),
    ]
if totals[2]:
    pooled = (1 - totals[0] / totals[1]) * 100
    word = "faster" if pooled > 0 else "slower"
    print(f"fixed-length sessions pooled over {totals[2]}: clockwork {word} by {abs(pooled):.0f}%")
    derived.append(("session_pooled_change_pct", f"{pooled:.1f}"))
with (REPO / "results" / "derived_metrics.csv").open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    writer.writerows(derived)
print("wrote results/derived_metrics.csv")

## 6b. Cross-check with vLLM's own benchmark harness

Sections 5 and 6 drive both engines from scripts/run_bench.py, which this repo wrote, so
every comparison so far rests on trusting that harness. This section removes the
objection. It clones the vLLM source tree at the installed version (benchmark_serving.py
ships in the source tree, not in the wheel), then runs that script against the clockwork
server and against the vLLM server in turn, same dataset, same request rate, same prompt
count, driven by a tool written by the baseline's authors. Both JSON result files are
saved under results/independent/ so they travel in the results zip, and the two are
printed side by side. Any failure prints a not-run line and the notebook continues.
Expected wall clock: 20 to 45 minutes, mostly server startup.

In [ ]:
INDEP_DIR = REPO / "results" / "independent"
INDEP_GIT = "https://github.com/vllm-project/vllm"
INDEPENDENT_STATUS = "not-run"
if vllm_python is None:
    INDEPENDENT_STATUS = "not-run: vllm is not installed"
    print("INDEPENDENT: not-run (vllm is not installed; section 6 printed the reason)")
else:
    try:
        INDEP_DIR.mkdir(parents=True, exist_ok=True)
        rc, lines = sh([vllm_python, "-c", "import vllm; print(vllm.__version__)"], echo=False)
        if rc != 0 or not lines:
            raise RuntimeError("cannot read vllm.__version__")
        vllm_version = lines[-1].strip()
        # benchmark_serving.py ships in the vllm source tree, not in the wheel, so the
        # harness has to come from a checkout of the version that is installed here.
        indep_src = BASE / "vllm_src"
        if not (indep_src / ".git").is_dir():
            tag = f"v{vllm_version}"
            clone = ["git", "clone", "--depth", "1", INDEP_GIT, str(indep_src)]
            tagged = ["git", "clone", "--depth", "1", "--branch", tag, INDEP_GIT, str(indep_src)]
            if sh(tagged)[0] != 0:
                shutil.rmtree(indep_src, ignore_errors=True)
                print(f"no tag {tag}; cloning vllm main, which may differ from the wheel")
                must(clone)
        rc, lines = sh(["git", "-C", str(indep_src), "log", "--oneline", "-1"], echo=False)
        indep_commit = lines[-1] if rc == 0 and lines else "unknown"
        print(f"vllm wheel {vllm_version}; harness checkout {indep_commit}")
        # vllm 0.28 moved this benchmark into the cli and left the script as a
        # deprecation stub that exits nonzero, so try entrypoints in order of
        # preference and keep the first whose --help actually works.
        bench_cli = Path(vllm_python).parent / "vllm"
        candidates = []
        if bench_cli.exists():
            candidates.append([str(bench_cli), "bench", "serve"])
        candidates.append([vllm_python, "-m", "vllm.entrypoints.cli.main", "bench", "serve"])
        candidates += [
            [vllm_python, str(path)]
            for path in sorted(indep_src.rglob("benchmark_serving.py"), key=lambda p: len(p.parts))
        ]
        indep_cmd, indep_help, tried = None, "", []
        for cand in candidates:
            rc, lines = sh(cand + ["--help"], echo=False)
            text = "\n".join(lines)
            if rc == 0 and "--num-prompts" in text:
                indep_cmd, indep_help = cand, text
                break
            tried.append(f"{' '.join(cand)} -> rc={rc}")
        if indep_cmd is None:
            raise RuntimeError("no working benchmark_serving entrypoint: " + "; ".join(tried))
        print("harness entrypoint:", " ".join(indep_cmd))

        def indep_has(flag):
            """Flag names drift across vllm releases; only pass what this copy accepts."""
            return flag in indep_help

        if SHAREGPT_OK and indep_has("--dataset-name"):
            indep_data = ["--dataset-name", "sharegpt", "--dataset-path", str(SHAREGPT_PATH)]
            indep_source = "sharegpt, " + str(SHAREGPT_PATH)
        elif SHAREGPT_OK and indep_has("--dataset"):
            indep_data = ["--dataset", str(SHAREGPT_PATH)]
            indep_source = "sharegpt via the legacy --dataset flag, " + str(SHAREGPT_PATH)
        elif indep_has("--dataset-name"):
            indep_data = ["--dataset-name", "random"]
            if indep_has("--random-input-len"):
                indep_data += ["--random-input-len", "512", "--random-output-len", "128"]
            indep_source = "SYNTHETIC vllm random dataset; the sharegpt file was unavailable"
        else:
            raise RuntimeError("benchmark_serving accepts neither --dataset-name nor --dataset")
        pool = [cfg for cfg in WORKLOADS if cfg.name in DATASET_WORKLOADS]
        ref = min(pool, key=lambda cfg: abs(cfg.request_rate - 4.0)) if pool else None
        indep_prompts = ref.num_requests if ref else 128
        indep_rate = ref.request_rate if ref else 4.0
        print(f"dataset: {indep_source}")
        print(f"{indep_prompts} prompts at {indep_rate:g} req/s against each engine in turn")

        def run_independent(label, server_cmd, base_url, log_name):
            """Serve one engine and drive it with vllm's own benchmark_serving.py."""
            out_path = INDEP_DIR / f"{label}.json"
            cmd = indep_cmd + [
                "--backend" if indep_has("--backend") else "--endpoint-type",
                "openai",
                "--base-url",
                base_url,
                "--endpoint",
                "/v1/completions",
                "--model",
                MODEL,
                "--num-prompts",
                str(indep_prompts),
                "--request-rate",
                str(indep_rate),
                "--save-result",
                *indep_data,
            ]
            scan = Path.cwd()
            if indep_has("--result-dir"):
                cmd += ["--result-dir", str(INDEP_DIR)]
                scan = INDEP_DIR
            if indep_has("--result-filename"):
                cmd += ["--result-filename", out_path.name]
            if indep_has("--seed"):
                cmd += ["--seed", "0"]
            before = {p.name for p in scan.glob("*.json")}
            proc = start_server(server_cmd, LOGS / log_name, f"{base_url}/health", timeout_s=1800)
            try:
                rc, _ = sh(cmd)
            finally:
                stop_server(proc)
            if rc != 0:
                raise RuntimeError(f"{label}: benchmark_serving exited {rc}")
            if not out_path.exists():
                # Older copies name the file themselves; take the one this run wrote.
                fresh = [p for p in scan.glob("*.json") if p.name not in before]
                if not fresh:
                    raise RuntimeError(f"{label}: benchmark_serving wrote no result json")
                max(fresh, key=lambda p: p.stat().st_mtime).replace(out_path)
            print(f"{label}: wrote {out_path}")
            return json.loads(out_path.read_text()), " ".join(cmd)

        clock_json, clock_cmd = run_independent(
            "clockwork", clockwork_cmd, CLOCKWORK_URL, "independent_clockwork.log"
        )
        vllm_json, vllm_indep_cmd = run_independent(
            "vllm", vllm_cmd, VLLM_URL, "independent_vllm.log"
        )
        shared = [
            key
            for key, value in clock_json.items()
            if isinstance(value, (int, float))
            and not isinstance(value, bool)
            and isinstance(vllm_json.get(key), (int, float))
            and not isinstance(vllm_json.get(key), bool)
        ]
        width = max(len(key) for key in shared)
        print(f"{'metric':<{width}}  {'clockwork':>14}  {'vllm':>14}  {'ratio':>8}")
        for key in shared:
            ours, theirs = float(clock_json[key]), float(vllm_json[key])
            ratio = f"{ours / theirs:.3f}" if theirs else ""
            print(f"{key:<{width}}  {ours:>14.3f}  {theirs:>14.3f}  {ratio:>8}")
        print("ratio is clockwork over vllm, both measured by the harness above;")
        print("throughput rows are higher-better, ttft, tpot and itl rows are lower-better")
        (INDEP_DIR / "provenance.json").write_text(
            json.dumps(
                {
                    "vllm_version": vllm_version,
                    "harness_repo": INDEP_GIT,
                    "harness_commit": indep_commit,
                    "harness_command": " ".join(indep_cmd),
                    "dataset": indep_source,
                    "num_prompts": indep_prompts,
                    "request_rate": indep_rate,
                    "clockwork_command": clock_cmd,
                    "vllm_command": vllm_indep_cmd,
                },
                indent=2,
            )
            + "\n"
        )
        INDEPENDENT_STATUS = f"ran: vllm {vllm_version} benchmark_serving.py against both servers"
        print("INDEPENDENT:", INDEPENDENT_STATUS)
    except Exception as exc:
        INDEPENDENT_STATUS = f"not-run: {exc}"
        print(f"INDEPENDENT: not-run ({exc})")

## 7. SGLang baseline

Attempts the same install and serve flow for SGLang. Any failure prints
SGLANG: not-run with the reason, and the run continues; SGLang depends on kernels that
may not support the T4's sm75. Expected wall clock: 2 to 30 minutes.

In [ ]:
SGLANG_STATUS = "not-run"
SGLANG_URL = "http://127.0.0.1:8200"
sglang_proc = None
try:
    rc, dry_lines = sh(
        [sys.executable, "-m", "pip", "install", "--dry-run", "sglang[all]"], echo=False
    )
    would = next((line for line in dry_lines if line.startswith("Would install")), "")
    conflict = re.search(r"\b(torch|transformers)-\d", would)
    if rc == 0 and not conflict:
        must([sys.executable, "-m", "pip", "install", "-q", "sglang[all]"])
        sglang_python = sys.executable
        print("SGLANG: installed in the current environment")
    else:
        if not make_env(BASE / "venv_sglang"):
            raise RuntimeError("could not create a venv for sglang")
        sglang_python = str(BASE / "venv_sglang" / "bin" / "python")
        must([sglang_python, "-m", "pip", "install", "-q", "sglang[all]"])
        print("SGLANG: installed in a fresh venv")
    sglang_cmd = [
        sglang_python,
        "-m",
        "sglang.launch_server",
        "--model-path",
        MODEL,
        "--dtype",
        "float16",
        "--port",
        "8200",
    ]
    sglang_proc = start_server(
        sglang_cmd, LOGS / "sglang_server.log", f"{SGLANG_URL}/health", timeout_s=1800
    )
    sglang_bench_cmd = [
        sys.executable,
        "scripts/run_bench.py",
        "--configs",
        "all",
        "--base-url",
        SGLANG_URL,
        "--out",
        "results/sglang",
    ]
    must(sglang_bench_cmd)
    stop_server(sglang_proc)
    SGLANG_STATUS = "ran: " + " ".join(sglang_bench_cmd)
    print("SGLANG: ran")
except Exception as exc:
    SGLANG_STATUS = f"not-run: {exc}"
    print(f"SGLANG: not-run ({exc})")
    if sglang_proc is not None and sglang_proc.poll() is None:
        sglang_proc.kill()

## 8. Results

Renders figures from every summary.csv through the repo's plotting entry point (clockwork
figures at docs/figures/, baselines in per-engine subdirectories), runs
scripts/collect_results.py on each engine's result directory for the markdown tables,
writes the GPU environment beside the CSVs, and zips results/ and docs/figures/ under
/content. Expected wall clock: under 2 minutes.

In [ ]:
from clockwork.bench.plots import plot_all

fig_base = REPO / "docs" / "figures"
for engine in ("clockwork", "vllm", "sglang"):
    summary = REPO / "results" / engine / "summary.csv"
    if not summary.exists():
        print(f"{engine}: no summary.csv, no figures")
        continue
    fig_dir = fig_base if engine == "clockwork" else fig_base / engine
    for figure in plot_all(summary, fig_dir):
        print("figure:", figure)

In [ ]:
env_path = REPO / "results" / "gpu_env.json"
env_path.write_text(
    json.dumps(
        {
            "gpu_name": GPU_NAME,
            "gpu_memory_gb": round(gpu_props.total_memory / 1e9, 2),
            "compute_capability": f"sm{gpu_props.major}{gpu_props.minor}",
            "cuda": torch.version.cuda,
        },
        indent=2,
    )
    + "\n"
)
print("environment:", env_path)
for engine in ("clockwork", "vllm", "sglang"):
    out_dir = REPO / "results" / engine
    if not (out_dir / "summary.csv").exists():
        print(f"{engine}: no summary.csv, no table")
        continue
    print(f"{engine} results:")
    must([sys.executable, "scripts/collect_results.py", "--out", str(out_dir), "--no-figures"])
zip_path = BASE / "clockwork_results_t4.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for tree in (REPO / "results", REPO / "docs" / "figures"):
        for file in sorted(p for p in tree.rglob("*") if p.is_file()):
            zf.write(file, file.relative_to(REPO))
print("results zip:", zip_path)

### Resume numbers

Prints rows formatted for private/RESUME_NUMBERS.md. Every value is read back from files
written earlier in this run and every row names the GPU.

In [ ]:
def read_summary(engine):
    path = REPO / "results" / engine / "summary.csv"
    if not path.exists():
        return []
    with path.open(newline="") as fh:
        return list(csv.DictReader(fh))


def fnum(row, key):
    value = row.get(key, "")
    return float(value) if value not in ("", None) else float("nan")


rows = []
for label, cmd, took in GATE_RESULTS:
    rows.append((f"{label} ({GPU_NAME})", f"PASS, {took:.0f}s", cmd))
best_micro = max(MICRO_ROWS, key=lambda row: row["speedup"])
rows.append(
    (
        f"triton vs torch paged decode ({GPU_NAME})",
        f"winner {BACKEND_WINNER}; best speedup {best_micro['speedup']:.2f}x "
        f"at batch {best_micro['batch']}, ctx {best_micro['ctx_len']}",
        "notebooks/bench_t4.ipynb microbench cell; results/microbench_decode.csv",
    )
)
clock = read_summary("clockwork")
by_name = {row["workload"]: row for row in clock}
if clock:
    peak = max(clock, key=lambda row: fnum(row, "output_tok_s"))
    rows.append(
        (
            f"clockwork peak output tok/s ({GPU_NAME})",
            f"{fnum(peak, 'output_tok_s'):.1f} on {peak['workload']}",
            CLOCKWORK_BENCH_CMDS[0],
        )
    )
    # Read the mid-rate single-turn workload out of the config list instead of
    # typing a name, so a rename drops neither this row nor its honesty.
    single = [cfg for cfg in WORKLOADS if cfg.kind not in ("agent", "ablation")]
    mid_name = min(single, key=lambda cfg: abs(cfg.request_rate - 4.0)).name if single else ""
    mid = by_name.get(mid_name)
    if mid:
        rows.append(
            (
                f"clockwork latency on {mid_name} ({GPU_NAME})",
                f"ttft p50 {fnum(mid, 'ttft_p50_ms'):.1f} ms, "
                f"p99 {fnum(mid, 'ttft_p99_ms'):.1f} ms, "
                f"itl p50 {fnum(mid, 'itl_p50_ms'):.1f} ms",
                CLOCKWORK_BENCH_CMDS[0],
            )
        )
    for rate in ("2", "8"):
        on = by_name.get(f"ablation_p1536_radix_on_r{rate}")
        off = by_name.get(f"ablation_p1536_radix_off_r{rate}")
        if on and off:
            rows.append(
                (
                    f"radix ablation at {rate} req/s ({GPU_NAME})",
                    f"ttft p50 {fnum(on, 'ttft_p50_ms'):.1f} ms on vs "
                    f"{fnum(off, 'ttft_p50_ms'):.1f} ms off; "
                    f"hit rate {fnum(on, 'hit_rate'):.2f}",
                    "; ".join(CLOCKWORK_BENCH_CMDS),
                )
            )
vllm_summary = read_summary("vllm")
vllm_by_name = {row["workload"]: row for row in vllm_summary}
common = sorted(set(by_name) & set(vllm_by_name))
if common:
    ours = sum(fnum(by_name[name], "output_tok_s") for name in common) / len(common)
    theirs = sum(fnum(vllm_by_name[name], "output_tok_s") for name in common) / len(common)
    rows.append(
        (
            f"clockwork vs vllm mean output tok/s, {len(common)} shared workloads ({GPU_NAME})",
            f"{ours:.1f} vs {theirs:.1f}",
            VLLM_BENCH_CMD,
        )
    )
rows.append((f"sglang baseline ({GPU_NAME})", SGLANG_STATUS, "notebooks/bench_t4.ipynb section 7"))
print("| value | measurement | source command |")
print("| --- | --- | --- |")
for value, measurement, source in rows:
    print(f"| {value} | {measurement} | {source} |")
print()
print("results zip:", ", ".join(str(p) for p in BASE.glob("clockwork_results_*.zip")))